In [59]:
import numpy as np
import pandas as pd

In [60]:

# 1. Define stock list
stocks = [
    "HDFCBANK", "ICICIBANK", "SBIN", "KOTAKBANK", "AXISBANK",
    "TCS", "INFY", "WIPRO", "HCLTECH", "TECHM",
    "SUNPHARMA", "DRREDDY", "CIPLA", "DIVISLAB", "APOLLOHOSP"
]

# 2. Sector mapping
sector_map = {
    "HDFCBANK": "Banking",
    "ICICIBANK": "Banking",
    "SBIN": "Banking",
    "KOTAKBANK": "Banking",
    "AXISBANK": "Banking",
    "TCS": "IT",
    "INFY": "IT",
    "WIPRO": "IT",
    "HCLTECH": "IT",
    "TECHM": "IT",
    "SUNPHARMA": "Pharma",
    "DRREDDY": "Pharma",
    "CIPLA": "Pharma",
    "DIVISLAB": "Pharma",
    "APOLLOHOSP": "Pharma"
}

# 3. Portfolio A: equal weights
equal_weight = 1 / len(stocks)

portfolio_a = pd.DataFrame({
    "Ticker": stocks,
    "Weight": equal_weight
})

portfolio_a["Weight (%)"] = portfolio_a["Weight"] * 100
portfolio_a["Sector"] = portfolio_a["Ticker"].map(sector_map)

# 4. Check total weight
print(portfolio_a)
print("\nTotal weight:", portfolio_a["Weight"].sum())
print("Total weight (%):", portfolio_a["Weight (%)"].sum())

# 5. Sector exposure in Portfolio A
portfolio_a_sector = portfolio_a.groupby("Sector")["Weight"].sum().reset_index()
portfolio_a_sector["Weight (%)"] = portfolio_a_sector["Weight"] * 100

print("\nPortfolio A sector exposure:")
print(portfolio_a_sector)

portfolio_a.to_csv("portfolio_a.csv", index=False)
print("\nSaved file: portfolio_a.csv")

        Ticker    Weight  Weight (%)   Sector
0     HDFCBANK  0.066667    6.666667  Banking
1    ICICIBANK  0.066667    6.666667  Banking
2         SBIN  0.066667    6.666667  Banking
3    KOTAKBANK  0.066667    6.666667  Banking
4     AXISBANK  0.066667    6.666667  Banking
5          TCS  0.066667    6.666667       IT
6         INFY  0.066667    6.666667       IT
7        WIPRO  0.066667    6.666667       IT
8      HCLTECH  0.066667    6.666667       IT
9        TECHM  0.066667    6.666667       IT
10   SUNPHARMA  0.066667    6.666667   Pharma
11     DRREDDY  0.066667    6.666667   Pharma
12       CIPLA  0.066667    6.666667   Pharma
13    DIVISLAB  0.066667    6.666667   Pharma
14  APOLLOHOSP  0.066667    6.666667   Pharma

Total weight: 0.9999999999999999
Total weight (%): 100.00000000000003

Portfolio A sector exposure:
    Sector    Weight  Weight (%)
0  Banking  0.333333   33.333333
1       IT  0.333333   33.333333
2   Pharma  0.333333   33.333333

Saved file: portfolio_a.csv


In [61]:
# 1. Load cleaned data
cleaned_data = pd.read_csv("cleaned_market_data.csv", parse_dates=["Date"])

# 2. Sort properly
cleaned_data = cleaned_data.sort_values(["Ticker", "Date"]).reset_index(drop=True)

# 3. Compute daily returns from Adjusted Close
cleaned_data["Daily Return"] = cleaned_data.groupby("Ticker")["Adj Close"].pct_change()

returns_data = cleaned_data[["Date", "Ticker", "Adj Close", "Daily Return"]].copy()

# 2. Separate stocks and benchmark
stock_returns = returns_data[returns_data["Ticker"] != "NIFTY50"].copy()

In [62]:
TRADING_DAYS = 252
RISK_FREE_RATE = 0.065

# 1. Create daily return matrix for 15 stocks
portfolio_returns_matrix = stock_returns.pivot(
    index="Date",
    columns="Ticker",
    values="Daily Return"
)

print("Return matrix shape:", portfolio_returns_matrix.shape)
print(portfolio_returns_matrix.head())

Return matrix shape: (491, 15)
Ticker      APOLLOHOSP  AXISBANK     CIPLA  DIVISLAB   DRREDDY   HCLTECH  \
Date                                                                       
2023-01-02         NaN       NaN       NaN       NaN       NaN       NaN   
2023-01-03    0.008205  0.021984  0.004622  0.006034  0.000779  0.001828   
2023-01-04   -0.012826 -0.005040 -0.008365  0.012541  0.003575 -0.004561   
2023-01-05   -0.000959 -0.008251  0.020105  0.007655  0.010344  0.007235   
2023-01-06   -0.009392 -0.010163 -0.011026 -0.007785 -0.005992 -0.010823   

Ticker      HDFCBANK  ICICIBANK      INFY  KOTAKBANK      SBIN  SUNPHARMA  \
Date                                                                        
2023-01-02       NaN        NaN       NaN        NaN       NaN        NaN   
2023-01-03  0.006539  -0.001219 -0.000952   0.004274  0.000327   0.012136   
2023-01-04 -0.017873  -0.002496 -0.018226  -0.006602 -0.011757  -0.004955   
2023-01-05 -0.006428  -0.022189 -0.013112  -0.00024

In [63]:
# 2. Equal weights aligned to matrix columns
weights_a = pd.Series(equal_weight, index=portfolio_returns_matrix.columns)

# 3. Portfolio A daily return
portfolio_a_daily = (portfolio_returns_matrix * weights_a).sum(axis=1)

# 4. Put into dataframe for inspection
portfolio_a_returns = pd.DataFrame({
    "Date": portfolio_a_daily.index,
    "Portfolio_A_Daily_Return": portfolio_a_daily.values
})

print(portfolio_a_returns.head(10))
print("\nShape:", portfolio_a_returns.shape)
print("\nMissing values:")
print(portfolio_a_returns.isna().sum())

        Date  Portfolio_A_Daily_Return
0 2023-01-02                  0.000000
1 2023-01-03                  0.006947
2 2023-01-04                 -0.006501
3 2023-01-05                 -0.000713
4 2023-01-06                 -0.012531
5 2023-01-09                  0.016236
6 2023-01-10                 -0.006599
7 2023-01-11                 -0.002254
8 2023-01-12                 -0.000858
9 2023-01-13                  0.004837

Shape: (491, 2)

Missing values:
Date                        0
Portfolio_A_Daily_Return    0
dtype: int64


In [64]:
# 5. Portfolio A metrics
mean_daily_a = portfolio_a_returns["Portfolio_A_Daily_Return"].mean()
std_daily_a = portfolio_a_returns["Portfolio_A_Daily_Return"].std()

portfolio_a_annual_return = mean_daily_a * TRADING_DAYS * 100
portfolio_a_annual_vol = std_daily_a * np.sqrt(TRADING_DAYS) * 100
portfolio_a_sharpe = ((mean_daily_a * TRADING_DAYS) - RISK_FREE_RATE) / (std_daily_a * np.sqrt(TRADING_DAYS))

print("Portfolio A Annualised Return (%):", round(portfolio_a_annual_return, 2))
print("Portfolio A Annualised Volatility (%):", round(portfolio_a_annual_vol, 2))
print("Portfolio A Sharpe Ratio:", round(portfolio_a_sharpe, 3))

Portfolio A Annualised Return (%): 22.23
Portfolio A Annualised Volatility (%): 11.41
Portfolio A Sharpe Ratio: 1.379


In [65]:
# Remove the artificial first row where all constituent returns were NaN
portfolio_returns_matrix_clean = portfolio_returns_matrix.dropna(how="all").copy()

print("Old shape:", portfolio_returns_matrix.shape)
print("New shape:", portfolio_returns_matrix_clean.shape)
print(portfolio_returns_matrix_clean.head())

Old shape: (491, 15)
New shape: (490, 15)
Ticker      APOLLOHOSP  AXISBANK     CIPLA  DIVISLAB   DRREDDY   HCLTECH  \
Date                                                                       
2023-01-03    0.008205  0.021984  0.004622  0.006034  0.000779  0.001828   
2023-01-04   -0.012826 -0.005040 -0.008365  0.012541  0.003575 -0.004561   
2023-01-05   -0.000959 -0.008251  0.020105  0.007655  0.010344  0.007235   
2023-01-06   -0.009392 -0.010163 -0.011026 -0.007785 -0.005992 -0.010823   
2023-01-09    0.004615  0.020055  0.010313  0.007162  0.014572  0.033550   

Ticker      HDFCBANK  ICICIBANK      INFY  KOTAKBANK      SBIN  SUNPHARMA  \
Date                                                                        
2023-01-03  0.006539  -0.001219 -0.000952   0.004274  0.000327   0.012136   
2023-01-04 -0.017873  -0.002496 -0.018226  -0.006602 -0.011757  -0.004955   
2023-01-05 -0.006428  -0.022189 -0.013112  -0.000247 -0.000165   0.012051   
2023-01-06 -0.003313  -0.010294 -0.01809

In [66]:
TRADING_DAYS = 252
RISK_FREE_RATE = 0.065

# Equal weights
weights_a = pd.Series(equal_weight, index=portfolio_returns_matrix_clean.columns)

# Portfolio A daily return
portfolio_a_daily = (portfolio_returns_matrix_clean * weights_a).sum(axis=1)

portfolio_a_returns = pd.DataFrame({
    "Date": portfolio_a_daily.index,
    "Portfolio_A_Daily_Return": portfolio_a_daily.values
})

print(portfolio_a_returns.head())
print("\nShape:", portfolio_a_returns.shape)
print("\nMissing values:")
print(portfolio_a_returns.isna().sum())

        Date  Portfolio_A_Daily_Return
0 2023-01-03                  0.006947
1 2023-01-04                 -0.006501
2 2023-01-05                 -0.000713
3 2023-01-06                 -0.012531
4 2023-01-09                  0.016236

Shape: (490, 2)

Missing values:
Date                        0
Portfolio_A_Daily_Return    0
dtype: int64


In [67]:
mean_daily_a = portfolio_a_returns["Portfolio_A_Daily_Return"].mean()
std_daily_a = portfolio_a_returns["Portfolio_A_Daily_Return"].std()

portfolio_a_annual_return = mean_daily_a * TRADING_DAYS * 100
portfolio_a_annual_vol = std_daily_a * np.sqrt(TRADING_DAYS) * 100
portfolio_a_sharpe = ((mean_daily_a * TRADING_DAYS) - RISK_FREE_RATE) / (std_daily_a * np.sqrt(TRADING_DAYS))

print("Portfolio A Annualised Return (%):", round(portfolio_a_annual_return, 2))
print("Portfolio A Annualised Volatility (%):", round(portfolio_a_annual_vol, 2))
print("Portfolio A Sharpe Ratio:", round(portfolio_a_sharpe, 3))

Portfolio A Annualised Return (%): 22.28
Portfolio A Annualised Volatility (%): 11.42
Portfolio A Sharpe Ratio: 1.382


In [68]:
df = pd.read_csv("task2_summary_table.csv")

In [69]:
# View key stock metrics for allocation decision
portfolio_b_view = df[[
    "Ticker",
    "Annualised Return (%)",
    "Annualised Volatility (%)",
    "Sharpe Ratio",
    "Beta vs NIFTY50",
    "Maximum Drawdown (%)"
]].copy()

# Add sector
portfolio_b_view["Sector"] = portfolio_b_view["Ticker"].map(sector_map)

print("Sorted by Sharpe Ratio:")
print(portfolio_b_view.sort_values("Sharpe Ratio", ascending=False).reset_index(drop=True))

print("\nSorted by Maximum Drawdown (%) (less negative is better):")
print(portfolio_b_view.sort_values("Maximum Drawdown (%)", ascending=False).reset_index(drop=True))

print("\nSorted by Beta vs NIFTY50:")
print(portfolio_b_view.sort_values("Beta vs NIFTY50").reset_index(drop=True))

Sorted by Sharpe Ratio:
        Ticker  Annualised Return (%)  Annualised Volatility (%)  \
0    SUNPHARMA                  35.34                      17.37   
1      HCLTECH                  37.86                      21.37   
2      DRREDDY                  28.03                      19.45   
3     DIVISLAB                  34.65                      26.26   
4        TECHM                  33.44                      25.16   
5   APOLLOHOSP                  28.03                      21.74   
6        WIPRO                  25.22                      24.15   
7    ICICIBANK                  20.59                      18.34   
8        CIPLA                  22.04                      24.07   
9          TCS                  16.41                      19.71   
10        SBIN                  18.59                      25.57   
11        INFY                  16.42                      23.09   
12    AXISBANK                   8.81                      21.90   
13    HDFCBANK          

In [70]:
portfolio_b = pd.DataFrame({
    "Ticker": [
        "SUNPHARMA", "HCLTECH", "DRREDDY", "DIVISLAB", "APOLLOHOSP",
        "ICICIBANK", "TECHM", "WIPRO", "CIPLA", "TCS",
        "SBIN", "INFY", "AXISBANK", "HDFCBANK", "KOTAKBANK"
    ],
    "Weight": [
        0.15, 0.12, 0.10, 0.10, 0.08,
        0.10, 0.08, 0.06, 0.05, 0.05,
        0.05, 0.03, 0.02, 0.01, 0.00
    ]
})

portfolio_b["Weight (%)"] = portfolio_b["Weight"] * 100
portfolio_b["Sector"] = portfolio_b["Ticker"].map(sector_map)

print(portfolio_b)
print("\nTotal weight:", portfolio_b["Weight"].sum())
print("Total weight (%):", portfolio_b["Weight (%)"].sum())

portfolio_b_sector = portfolio_b.groupby("Sector")["Weight"].sum().reset_index()
portfolio_b_sector["Weight (%)"] = portfolio_b_sector["Weight"] * 100

print("\nPortfolio B sector exposure:")
print(portfolio_b_sector)

portfolio_a.to_csv("portfolio_b.csv", index=False)
print("\nSaved file: portfolio_b.csv")

        Ticker  Weight  Weight (%)   Sector
0    SUNPHARMA    0.15        15.0   Pharma
1      HCLTECH    0.12        12.0       IT
2      DRREDDY    0.10        10.0   Pharma
3     DIVISLAB    0.10        10.0   Pharma
4   APOLLOHOSP    0.08         8.0   Pharma
5    ICICIBANK    0.10        10.0  Banking
6        TECHM    0.08         8.0       IT
7        WIPRO    0.06         6.0       IT
8        CIPLA    0.05         5.0   Pharma
9          TCS    0.05         5.0       IT
10        SBIN    0.05         5.0  Banking
11        INFY    0.03         3.0       IT
12    AXISBANK    0.02         2.0  Banking
13    HDFCBANK    0.01         1.0  Banking
14   KOTAKBANK    0.00         0.0  Banking

Total weight: 1.0000000000000002
Total weight (%): 100.0

Portfolio B sector exposure:
    Sector  Weight  Weight (%)
0  Banking    0.18        18.0
1       IT    0.34        34.0
2   Pharma    0.48        48.0

Saved file: portfolio_b.csv


In [71]:
TRADING_DAYS = 252
RISK_FREE_RATE = 0.065

# 1. Create weight series aligned to return matrix columns
weights_b = portfolio_b.set_index("Ticker")["Weight"].reindex(portfolio_returns_matrix_clean.columns)

print(weights_b)
print("\nTotal weight:", weights_b.sum())

Ticker
APOLLOHOSP    0.08
AXISBANK      0.02
CIPLA         0.05
DIVISLAB      0.10
DRREDDY       0.10
HCLTECH       0.12
HDFCBANK      0.01
ICICIBANK     0.10
INFY          0.03
KOTAKBANK     0.00
SBIN          0.05
SUNPHARMA     0.15
TCS           0.05
TECHM         0.08
WIPRO         0.06
Name: Weight, dtype: float64

Total weight: 1.0000000000000002


In [72]:
# 2. Portfolio B daily return
portfolio_b_daily = (portfolio_returns_matrix_clean * weights_b).sum(axis=1)

portfolio_b_returns = pd.DataFrame({
    "Date": portfolio_b_daily.index,
    "Portfolio_B_Daily_Return": portfolio_b_daily.values
})

print(portfolio_b_returns.head())
print("\nShape:", portfolio_b_returns.shape)
print("\nMissing values:")
print(portfolio_b_returns.isna().sum())

        Date  Portfolio_B_Daily_Return
0 2023-01-03                  0.006488
1 2023-01-04                 -0.004122
2 2023-01-05                  0.001392
3 2023-01-06                 -0.011542
4 2023-01-09                  0.016009

Shape: (490, 2)

Missing values:
Date                        0
Portfolio_B_Daily_Return    0
dtype: int64


In [73]:
# 3. Portfolio B metrics
mean_daily_b = portfolio_b_returns["Portfolio_B_Daily_Return"].mean()
std_daily_b = portfolio_b_returns["Portfolio_B_Daily_Return"].std()

portfolio_b_annual_return = mean_daily_b * TRADING_DAYS * 100
portfolio_b_annual_vol = std_daily_b * np.sqrt(TRADING_DAYS) * 100
portfolio_b_sharpe = ((mean_daily_b * TRADING_DAYS) - RISK_FREE_RATE) / (std_daily_b * np.sqrt(TRADING_DAYS))

print("Portfolio B Annualised Return (%):", round(portfolio_b_annual_return, 2))
print("Portfolio B Annualised Volatility (%):", round(portfolio_b_annual_vol, 2))
print("Portfolio B Sharpe Ratio:", round(portfolio_b_sharpe, 3))

Portfolio B Annualised Return (%): 28.2
Portfolio B Annualised Volatility (%): 11.65
Portfolio B Sharpe Ratio: 1.863


In [74]:
portfolio_comparison = pd.DataFrame({
    "Portfolio": ["Portfolio A", "Portfolio B"],
    "Annualised Return (%)": [round(portfolio_a_annual_return, 2), round(portfolio_b_annual_return, 2)],
    "Annualised Volatility (%)": [round(portfolio_a_annual_vol, 2), round(portfolio_b_annual_vol, 2)],
    "Sharpe Ratio": [round(portfolio_a_sharpe, 3), round(portfolio_b_sharpe, 3)]
})

print(portfolio_comparison)

     Portfolio  Annualised Return (%)  Annualised Volatility (%)  Sharpe Ratio
0  Portfolio A                  22.28                      11.42         1.382
1  Portfolio B                  28.20                      11.65         1.863


In [75]:
# Sector exposure comparison
portfolio_a_sector_exposure = portfolio_a.groupby("Sector")["Weight"].sum() * 100
portfolio_b_sector_exposure = portfolio_b.groupby("Sector")["Weight"].sum() * 100

sector_exposure_comparison = pd.DataFrame({
    "Sector": ["Banking", "IT", "Pharma"],
    "Portfolio A Exposure (%)": [
        portfolio_a_sector_exposure.get("Banking", 0),
        portfolio_a_sector_exposure.get("IT", 0),
        portfolio_a_sector_exposure.get("Pharma", 0)
    ],
    "Portfolio B Exposure (%)": [
        portfolio_b_sector_exposure.get("Banking", 0),
        portfolio_b_sector_exposure.get("IT", 0),
        portfolio_b_sector_exposure.get("Pharma", 0)
    ]
})

print(sector_exposure_comparison)

    Sector  Portfolio A Exposure (%)  Portfolio B Exposure (%)
0  Banking                 33.333333                      18.0
1       IT                 33.333333                      34.0
2   Pharma                 33.333333                      48.0


In [76]:
# Sector-level averages from stock metrics
sector_summary = df.copy()
sector_summary["Sector"] = sector_summary["Ticker"].map(sector_map)

sector_breakdown = sector_summary.groupby("Sector").agg(
    Average_Sharpe_Ratio=("Sharpe Ratio", "mean"),
    Average_Beta_vs_NIFTY50=("Beta vs NIFTY50", "mean")
).reset_index()

sector_breakdown["Average_Sharpe_Ratio"] = sector_breakdown["Average_Sharpe_Ratio"].round(3)
sector_breakdown["Average_Beta_vs_NIFTY50"] = sector_breakdown["Average_Beta_vs_NIFTY50"].round(3)

print(sector_breakdown)

    Sector  Average_Sharpe_Ratio  Average_Beta_vs_NIFTY50
0  Banking                 0.228                    1.065
1       IT                 0.849                    0.876
2   Pharma                 1.095                    0.450


In [77]:
sector_exposure_comparison.to_csv("task4_sector_exposure_comparison.csv", index=False)
sector_breakdown.to_csv("task4_sector_breakdown.csv", index=False)

print("Saved:")
print("- task4_sector_exposure_comparison.csv")
print("- task4_sector_breakdown.csv")

Saved:
- task4_sector_exposure_comparison.csv
- task4_sector_breakdown.csv
